![image_1780921115942.png](./image_1780921115942.png "image_1780921115942.png")

![image_1780921130924.png](./image_1780921130924.png "image_1780921130924.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql import functions as F
# Initialize Spark session
spark = SparkSession.builder.appName("PhoneCallsExample").getOrCreate()

# Define schema for phone_calls
phone_calls_schema = StructType([
    StructField("caller_id", IntegerType(), True),
    StructField("receiver_id", IntegerType(), True),
    StructField("call_time", StringType(), True)  # can later cast to TimestampType
])

# Create phone_calls DataFrame
phone_calls_data = [
    (10, 20, "2023-01-01 08:00:00"),
    (20, 30, "2023-01-01 09:00:00"),
    (30, 40, "2023-01-02 10:00:00"),
    (40, 10, "2023-01-02 11:00:00"),
    (10, 30, "2023-01-03 12:00:00"),
    (50, 60, "2023-01-03 13:00:00"),
    (60, 50, "2023-01-04 14:00:00"),
    (10, 50, "2023-01-04 15:00:00")
]

phone_calls_df = spark.createDataFrame(phone_calls_data, schema=phone_calls_schema)

# Define schema for phone_info
phone_info_schema = StructType([
    StructField("caller_id", IntegerType(), True),
    StructField("country_id", IntegerType(), True)
])

# Create phone_info DataFrame
phone_info_data = [
    (10, 100),
    (20, 100),
    (30, 200),
    (40, 200),
    (50, 100),
    (60, 300)
]

phone_info_df = spark.createDataFrame(phone_info_data, schema=phone_info_schema)

phone_calls_df = phone_calls_df.withColumn("call_time", F.to_timestamp("call_time"))
# Show the DataFrames

print("Phone Calls DataFrame:")
phone_calls_df.show()

print("Phone Info DataFrame:")
phone_info_df.show()


In [0]:
pc_df = phone_calls_df.alias("pc")
pf_df = phone_info_df.alias("pf")
po_df = phone_info_df.alias("po")

result_df = (
    pc_df.join(pf_df, F.col("pc.caller_id") == F.col("pf.caller_id"), "inner")
    .join(po_df, F.col("pc.receiver_id") == F.col("po.caller_id"), "inner")
    .select(
        F.round(
            (
                F.sum(
                    F.when(
                        F.col("pf.country_id") != F.col("po.country_id"), 1
                    ).otherwise(F.lit(0))
                )
                * 100
            )
            / F.count("*"),
            2,
        ).alias("international_calls_pct")
    )
)
display(result_df)